# Export the Deployable Model - Random Forest Pipeline

**Input:** `unsw_cleaned_features.csv` (2,059,414 rows x 42 columns).

Notebook 04 trained and evaluated Random Forest as one of the 4 candidates, using the 10-column feature set locked in by notebook 03. This notebook does one thing notebook 04 didn't: it bundles the **preprocessing and the model into a single artifact** - a scikit-learn `Pipeline` - and saves it to disk, so a separate program (the router simulation's Flask backend) can load it and call `.predict_proba()` directly, without reimplementing the one-hot encoding by hand.

Same feature set, same split, same hyperparameters as notebook 04 - this is not a retrain with different settings, it's packaging the same model for deployment.

## Contents

1. Load data, feature / target setup
2. Train / test split (same 60/20/20 as notebook 04)
3. Build and fit the deployable pipeline
4. Confirm it matches notebook 04's reported performance
5. Export the pipeline + a model info card
6. Sanity-check the exported file by loading it back

In [1]:
import json
import platform
import sklearn
import joblib
import pandas as pd
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)

DATA_PATH = 'unsw_cleaned_features.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
print('Shape:', df.shape)

Shape: (2059414, 42)


## Section 1 - Feature / target setup

The exact 10-column `final_feature_cols` locked in by notebook 03/04 - `smeansz` + `dst_port_bucket` + the 8 `ct_*` rolling counters. `Spkts`/`sbytes`/`sttl`/`swin`/`proto`/`service` stay excluded for the reasons documented there (dominance / TCP-identity leakage).

In [2]:
candidate_categorical = ['dst_port_bucket']
candidate_numeric = [
    'smeansz',
    'ct_state_ttl', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
    'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
]
final_feature_cols = candidate_numeric + candidate_categorical

X = df[final_feature_cols]
y = df['packet_lost']

print(f"Features: {len(candidate_numeric)} numeric + {len(candidate_categorical)} categorical = {X.shape[1]} total")
print(f"Target positive rate: {y.mean()*100:.2f}%")

Features: 9 numeric + 1 categorical = 10 total
Target positive rate: 70.31%


## Section 2 - Train / test split

Same 60% train / 20% validation / 20% test as notebook 04. The pipeline is fit on the 60% train split only; the held-out 20% test split confirms its performance matches what notebook 04 already reported before we trust it as the deployment artifact.

In [3]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print(f"Train shape: {X_train.shape}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test shape : {X_test.shape}  ({len(X_test)/len(X)*100:.1f}%)")

Train shape: (1235648, 10)  (60.0%)
Test shape : (411883, 10)  (20.0%)


## Section 3 - Build and fit the deployable pipeline

A single `Pipeline`: one-hot encode `dst_port_bucket`, pass the numeric columns through unchanged, then Random Forest - identical settings to notebook 04 (`n_estimators=200, max_depth=5, class_weight='balanced'`). Bundling preprocessing and model together means the backend only ever has to call `.predict_proba(raw_dataframe)` - it never needs to know the encoder existed.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

preprocessor = ColumnTransformer([
    ('num', 'passthrough', candidate_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), candidate_categorical),
])

model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
print('Pipeline fitted.')

Pipeline fitted.


## Section 4 - Confirm it matches notebook 04's reported performance

Same split, same settings - this should land at the same ~93-94% accuracy notebook 04 reported for Random Forest, not a new number.

In [5]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

test_preds = model.predict(X_test)
test_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_preds, average='macro', zero_division=0)
cm = confusion_matrix(y_test, test_preds)

print(f"Test accuracy : {acc*100:.2f}%")
print(f"Macro F1      : {f1:.4f}")
print('Confusion matrix:')
print(cm)

Test accuracy : 93.67%
Macro F1      : 0.9273
Confusion matrix:
[[119020   3260]
 [ 22815 266788]]


## Section 5 - Export the pipeline + a model info card

Two files, saved into this project folder - the router simulation's backend copies these into its own `backend/model/` folder as its deployment artifact, but this folder is the source of truth for how they were produced.

In [6]:
import os

os.makedirs('model', exist_ok=True)

MODEL_PATH = 'model/random_forest_pipeline.joblib'
joblib.dump(model, MODEL_PATH)
print(f"Saved pipeline -> {MODEL_PATH}")

Saved pipeline -> model/random_forest_pipeline.joblib


In [7]:
model_info = {
    'model_type': 'RandomForestClassifier (scikit-learn Pipeline)',
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'sklearn_version': sklearn.__version__,
    'python_version': platform.python_version(),
    'feature_columns': final_feature_cols,
    'categorical_columns': candidate_categorical,
    'numeric_columns': candidate_numeric,
    'hyperparameters': {
        'n_estimators': 200, 'max_depth': 5, 'class_weight': 'balanced', 'random_state': 42,
    },
    'training_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'test_accuracy_pct': round(float(acc) * 100, 2),
    'test_macro_f1': round(float(f1), 4),
    'test_confusion_matrix': cm.tolist(),
    'notes': (
        'Trained on IOT_Packet_Loss_ML_Model/unsw_cleaned_features.csv. '
        'Predicts packet_lost probability from 10 pre-send-safe features only - '
        'see notebooks 03-04 for why Spkts/sbytes/sttl/swin/proto/service were excluded.'
    ),
}

INFO_PATH = 'model/model_info.json'
with open(INFO_PATH, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2)
print(f"Saved model info -> {INFO_PATH}")
model_info

Saved model info -> model/model_info.json


{'model_type': 'RandomForestClassifier (scikit-learn Pipeline)',
 'exported_at_utc': '2026-09-16T09:10:47.743653+00:00',
 'sklearn_version': '1.9.0',
 'python_version': '3.12.10',
 'feature_columns': ['smeansz',
  'ct_state_ttl',
  'ct_srv_src',
  'ct_srv_dst',
  'ct_dst_ltm',
  'ct_src_ltm',
  'ct_src_dport_ltm',
  'ct_dst_sport_ltm',
  'ct_dst_src_ltm',
  'dst_port_bucket'],
 'categorical_columns': ['dst_port_bucket'],
 'numeric_columns': ['smeansz',
  'ct_state_ttl',
  'ct_srv_src',
  'ct_srv_dst',
  'ct_dst_ltm',
  'ct_src_ltm',
  'ct_src_dport_ltm',
  'ct_dst_sport_ltm',
  'ct_dst_src_ltm'],
 'hyperparameters': {'n_estimators': 200,
  'max_depth': 5,
  'class_weight': 'balanced',
  'random_state': 42},
 'training_rows': 1235648,
 'test_rows': 411883,
 'test_accuracy_pct': 93.67,
 'test_macro_f1': 0.9273,
 'test_confusion_matrix': [[119020, 3260], [22815, 266788]],
 'notes': 'Trained on IOT_Packet_Loss_ML_Model/unsw_cleaned_features.csv. Predicts packet_lost probability from 10 pre

## Section 6 - Sanity-check the exported file by loading it back

Load `random_forest_pipeline.joblib` fresh, as if this were the backend doing it, and confirm it gives the same predictions as the in-memory `model` above - the same check a client could run to prove the exported file isn't different from what was just evaluated.

In [8]:
reloaded = joblib.load(MODEL_PATH)
reloaded_proba = reloaded.predict_proba(X_test)[:, 1]

import numpy as np
identical = np.allclose(test_proba, reloaded_proba)
print(f"Reloaded model's predictions match the in-memory model exactly: {identical}")

sample = X_test.iloc[[0]]
print()
print('Example single-row call (this is exactly what the backend will do per packet):')
print(sample)
print('predict_proba ->', reloaded.predict_proba(sample)[0])

Reloaded model's predictions match the in-memory model exactly: True

Example single-row call (this is exactly what the backend will do per packet):
         smeansz  ct_state_ttl  ct_srv_src  ct_srv_dst  ct_dst_ltm  \
1077332       65             0           2           2           3   

         ct_src_ltm  ct_src_dport_ltm  ct_dst_sport_ltm  ct_dst_src_ltm  \
1077332           5                 1                 1               2   

        dst_port_bucket  
1077332      well_known  
predict_proba -> [0.83058434 0.16941566]
